# Player Number-Guided AI Highlight Video Curator
**CSCI 5922 — Mateusz Muszynski & Colin Wallace**

Tracks jersey **#6 (dark kit)** through *Denver vs. Virginia | 2025 ACC Men's Soccer*
and assembles a personal highlight reel.

### Before first run
1. **Settings (right panel) → Accelerator → GPU T4 x1**
2. **Settings → Internet → On** (needed to clone the repo)
3. Add the match video as a Kaggle dataset named `soccer-match-video` (see Cell 2)
4. Run cells top-to-bottom

### Re-running
Every cell is idempotent — already-done work is skipped.

---
## 0 — GPU check

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — open Settings → Accelerator → GPU T4, then re-run all')

---
## 1 — Clone / update repo & install dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/mateusz-muszynski/csci5922-highlight-curator.git'
REPO_DIR = '/kaggle/working/csci5922'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --rebase

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'Repo: {os.getcwd()}')
!git log --oneline -5

In [ ]:
!pip install -q \
    ultralytics>=8.2.0 \
    supervision>=0.21.0 \
    opencv-python-headless \
    albumentations \
    ffmpeg-python \
    imageio[ffmpeg] \
    PyYAML tqdm rich

import torch, torchvision, ultralytics, supervision, cv2
print(f'torch {torch.__version__} | tv {torchvision.__version__} | '
      f'ultralytics {ultralytics.__version__} | cv2 {cv2.__version__}')

---
## 2 — Configuration

### How to add the match video as a Kaggle dataset
1. kaggle.com → **Datasets** → **New Dataset**
2. Upload the mp4; name the dataset **`soccer-match-video`**
3. Back here: **Add data** (right panel) → search `soccer-match-video` → Add
4. File appears at `/kaggle/input/soccer-match-video/<filename.mp4>`
5. Update `VIDEO_PATH` below to match the exact filename

In [ ]:
import os

# --- EDIT THESE ---------------------------------------------------------
VIDEO_PATH    = "/kaggle/input/soccer-match-video/denver.mp4"
TARGET_JERSEY = 6
JERSEY_COLOR  = 'dark'   # jersey #6 wears a dark kit
# -------------------------------------------------------------------------

OUTPUT_DIR  = '/kaggle/working/outputs'
OUTPUT_PATH = f'{OUTPUT_DIR}/highlights_jersey{TARGET_JERSEY}.mp4'
CONFIG_PATH = os.path.join(REPO_DIR, 'config.yaml')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Available input files:')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(f'  {os.path.join(root, f)}')

assert os.path.exists(VIDEO_PATH), (
    f'Video not found: {VIDEO_PATH}\n'
    'Create a Kaggle dataset named soccer-match-video with the mp4 file.'
)
print(f'\nVideo : {os.path.basename(VIDEO_PATH)}')
print(f'Size  : {os.path.getsize(VIDEO_PATH)/1e9:.2f} GB')
print(f'Jersey: #{TARGET_JERSEY} ({JERSEY_COLOR} kit)')

---
## 3 — Trim a 10-minute gameplay clip
Skips any pre-game intro. Re-running is instant if the clip already exists.

In [ ]:
import os

START_SEC    = 300    # 5:00 — increase if there is a long intro
DURATION_SEC = 600   # 10 minutes of live play
CLIP_PATH    = f'/kaggle/working/game_clip_jersey{TARGET_JERSEY}.mp4'

if not os.path.exists(CLIP_PATH):
    print('Trimming clip...')
    !ffmpeg -y -ss {START_SEC} -i "{VIDEO_PATH}" -t {DURATION_SEC} \
        -c copy "{CLIP_PATH}" -loglevel error
    print(f'Clip saved: {os.path.getsize(CLIP_PATH)/1e6:.1f} MB')
else:
    print(f'Clip already exists ({os.path.getsize(CLIP_PATH)/1e6:.1f} MB) — skipping trim')

VIDEO_PATH = CLIP_PATH

---
## 4 — Build model weights + verify key alignment

Trains jersey CNN + LSTM scorer on synthetic stubs (~5 min on T4).
Skipped automatically if valid weights already exist.

**Key alignment check**: The weight file must contain keys starting with `backbone.`
(e.g. `backbone.conv1.weight`). If keys are wrong the model silently runs on random
weights — this cell catches that before the pipeline runs.

In [ ]:
import os, shutil, torch
os.chdir(REPO_DIR)

WEIGHTS_OUT_DIR = "/kaggle/working/models"   # persists in Output tab
MODELS_DIR      = os.path.join(REPO_DIR, "models")
os.makedirs(WEIGHTS_OUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

jersey_weights = os.path.join(MODELS_DIR, "jersey_cnn.pt")
scorer_weights = os.path.join(MODELS_DIR, "scorer_lstm.pt")

# ── Load from previously-saved output dataset (skip retraining) ───────────────
INPUT_WEIGHTS_DIR = "/kaggle/input/csci5922-weights"
for fname in ("jersey_cnn.pt", "scorer_lstm.pt"):
    src_path = os.path.join(INPUT_WEIGHTS_DIR, fname)
    dst_path = os.path.join(MODELS_DIR, fname)
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        shutil.copy(src_path, dst_path)
        print(f"Loaded pre-saved weights: {fname}")


def _weights_valid(path, prefix="backbone."):
    if not os.path.exists(path):
        return False
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
        keys  = list(state.keys())
        return len(keys) > 0 and any(k.startswith(prefix) for k in keys)
    except Exception as exc:
        print(f"  [WARN] {path}: {exc}")
        return False


jersey_ok = _weights_valid(jersey_weights)
scorer_ok = _weights_valid(scorer_weights)

if not jersey_ok or not scorer_ok:
    missing = [s for s, ok in [("jersey", jersey_ok), ("scorer", scorer_ok)] if not ok]
    print(f"Training ({missing}) — ResNet-34 on 100-class PIL-rendered digits ...")
    print("  Jersey CNN : 100 classes × 500 images, 20 epochs  (~12 min on T4)")
    print("  Scorer LSTM: 300 clips × 32 frames,   15 epochs  (~ 5 min on T4)")
    !python run_training.py --kaggle --stages jersey scorer --device cuda --fail-fast
    print()
    print("─── KAGGLE-FULL OPTION ─────────────────────────────────────────────────")
    print("For real SoccerNet accuracy, register at https://www.soccer-net.org/ then run:")
    print("  !SOCCERNET_USER=your@email SOCCERNET_PASS=yourpw \\")
    print("     python scripts/download_soccernet.py --task jersey --data-dir data/")
    print("  !python run_training.py --kaggle-full --stages jersey scorer --device cuda")
    print("────────────────────────────────────────────────────────────────────────")
else:
    print("Weights valid — skipping training")

# Hard stop if training failed
assert _weights_valid(jersey_weights), "jersey_cnn.pt missing or wrong keys"
assert _weights_valid(scorer_weights), "scorer_lstm.pt missing or wrong keys"

# Copy to /kaggle/working/models/ so they show in Output tab
for fname in ("jersey_cnn.pt", "scorer_lstm.pt"):
    shutil.copy(os.path.join(MODELS_DIR, fname), os.path.join(WEIGHTS_OUT_DIR, fname))

# Print key alignment report
j_state = torch.load(jersey_weights, map_location="cpu", weights_only=True)
s_state = torch.load(scorer_weights, map_location="cpu", weights_only=True)
print(f"jersey_cnn.pt  : {os.path.getsize(jersey_weights)/1e6:.1f} MB | first key: {list(j_state.keys())[0]!r}")
print(f"scorer_lstm.pt : {os.path.getsize(scorer_weights)/1e6:.1f} MB | first key: {list(s_state.keys())[0]!r}")
print("\nKey alignment OK — weights load cleanly")
print(f"Weights saved to Output tab → {WEIGHTS_OUT_DIR}/")
print("TIP: Save Output as dataset \'csci5922-weights\' to skip retraining next run.")


---
## 5 — Run the full pipeline

In [ ]:
import os, sys
os.chdir(REPO_DIR)

for mod in list(sys.modules.keys()):
    if mod.startswith('src') or mod == 'main':
        del sys.modules[mod]

from main import run_pipeline

DEBUG_VIDEO = f'/kaggle/working/debug_jersey{TARGET_JERSEY}.mp4'

result = run_pipeline(
    video_path=VIDEO_PATH,
    jersey_number=TARGET_JERSEY,
    output_path=OUTPUT_PATH,
    config_path=CONFIG_PATH,
    device='cuda',
    conf_override=0.25,
    highlight_thresh_override=0.40,
    frame_skip=1,
    clip_length_override=90,
    clip_stride_override=15,
    jersey_color=JERSEY_COLOR,
    debug_video=DEBUG_VIDEO,
)
print(f'\nHighlight reel -> {result}')

---
## 6 — Preview debug overlay
Green box = jersey #6. All tracked players labelled.

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

PREVIEW = f'/kaggle/working/debug_preview_jersey{TARGET_JERSEY}.mp4'
!ffmpeg -y -i "{DEBUG_VIDEO}" -vcodec libx264 -acodec aac \
    -vf 'scale=960:-2' -crf 28 -preset fast "{PREVIEW}" -loglevel error

data = open(PREVIEW, 'rb').read()
b64  = b64encode(data).decode()
display(HTML(f'''
<h3>Debug overlay — jersey #{TARGET_JERSEY} in green</h3>
<video width='960' controls>
  <source src='data:video/mp4;base64,{b64}' type='video/mp4'>
</video>
'''))

---
## 7 — Preview highlight reel

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

if result and os.path.exists(result):
    REEL = f'/kaggle/working/reel_preview_jersey{TARGET_JERSEY}.mp4'
    !ffmpeg -y -i "{result}" -vcodec libx264 -acodec aac \
        -crf 26 -preset fast "{REEL}" -loglevel error
    data = open(REEL, 'rb').read()
    b64  = b64encode(data).decode()
    display(HTML(f'''
    <h3>Highlight reel — jersey #{TARGET_JERSEY}</h3>
    <video width='960' controls>
      <source src='data:video/mp4;base64,{b64}' type='video/mp4'>
    </video>
    '''))
else:
    print(f'No highlights found for jersey #{TARGET_JERSEY}.')
    print('Try lowering highlight_thresh_override to 0.30 in Cell 5 and re-running.')
    print('Or run Cell 8 to check which numbers the model reads with the dark filter.')

---
## 8 — Jersey frequency scan
Samples 300 frames with the dark-kit filter active to verify #6 is detected.

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
import os, sys
os.chdir(REPO_DIR)

from src.detector import PlayerDetector
from src.jersey_reader import JerseyReader

det = PlayerDetector(conf_threshold=0.25, device='cuda')
jr  = JerseyReader(
    model_path=os.path.join(REPO_DIR, 'models/jersey_cnn.pt'),
    color_hint=JERSEY_COLOR,
    device='cuda'
)

cap     = cv2.VideoCapture(VIDEO_PATH)
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
counter = Counter()
indices = np.linspace(0, total - 1, min(300, max(1, total // 10)), dtype=int)

for fi in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
    ret, frame = cap.read()
    if not ret:
        continue
    for d in det.detect(frame):
        crop = det.crop_upper_body(frame, d)
        num, conf = jr.read(crop)
        if num is not None:
            counter[num] += 1
cap.release()

df = pd.DataFrame(counter.most_common(20), columns=['jersey', 'sightings'])
print(df.to_string(index=False))
print(f'\n#{TARGET_JERSEY} sightings: {counter.get(TARGET_JERSEY, 0)} / {len(indices)} sampled frames')

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(df['jersey'].astype(str), df['sightings'], color='steelblue')
for bar, jersey in zip(bars, df['jersey']):
    if jersey == TARGET_JERSEY:
        bar.set_color('green')
ax.set_xlabel('Jersey number')
ax.set_ylabel('Sightings (dark-kit filter active)')
ax.set_title(f'Jersey frequency scan — #{TARGET_JERSEY} in green')
plt.tight_layout()
plt.savefig(f'/kaggle/working/jersey_freq_{TARGET_JERSEY}.png', dpi=150)
plt.show()